# Phase 3 - Measure the Cost of Restarting

**RHOAIENG-80952**

## What this notebook does

Every time a training job resizes (GPUs added or removed), PyTorch kills all
workers and restarts from the last checkpoint. This notebook measures how long
that restart takes and where the time goes.

The experiment:

1. Run a training job on 8 GPUs until it saves the first checkpoint at `save_steps`
2. Kill the job (simulating Kueue preemption)
3. Submit the same job again - it resumes from the checkpoint automatically
4. Measure the restart startup timeline and check loss after resume

## Why this is the most important measurement

The restart cost is the one number that decides whether elastic scaling can
ever pay for itself. If restarting takes 2 minutes, scaling up is worth it
whenever more than ~10 minutes of work remain. If it takes 20 minutes, elastic
scaling only helps very long jobs.

The research doc (H2) predicts that saving the checkpoint is cheap (LoRA
adapters are small) but restarting is expensive (container startup, model
loading, rendezvous). This experiment finds out exactly how expensive.

### Prerequisites

- IBM cluster workbench in `dhryshch-elastic-scaling` namespace
- PVC `elastic-scaling-shared` and Secret `hf-token`
- 8 GPUs available on the target node

Training code lives in `phase3/train.py`.

## Setup

In [ ]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"
!pip install yamlmagic hatchling --index-url https://pypi.org/simple
!pip install --no-deps .. --index-url https://pypi.org/simple
%load_ext yamlmagic

## Configuration

Same model, dataset, and training settings as Phase 1. The key difference
is that `save_steps` creates checkpoints we can restart from.

In [ ]:
%%yaml parameters

# Infrastructure
namespace: dhryshch-elastic-scaling
pvc_name: elastic-scaling-shared
hf_secret_name: hf-token
target_node: oai-kft-ibm-jcsbk-gpu-2-8gmgw

# Model & Data
model_id: meta-llama/Llama-3.1-8B
dataset_id: tatsu-lab/alpaca
output_dir: /mnt/kubeflow-checkpoints

# Training
max_steps: 200
save_steps: 100
seq_length: 1024
seed: 42
global_batch_size: 128
per_device_batch_size: 4
lora_r: 16
lora_alpha: 32
warmup_steps_excluded: 20

# Phase 3 specific
num_nodes: 1
gpus: 8

In [ ]:
%load_ext autoreload
%autoreload 2

from elastic_scaling_poc.phase3.train import train_func

print("train_func loaded")

In [ ]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

api_server = os.environ["OPENSHIFT_API_URL"]
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not token:
    sa_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_path.exists():
        token = sa_path.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench "
        "with a service-account token."
    )

config = k8s.Configuration()
config.host = api_server
config.api_key = {"authorization": f"Bearer {token}"}
config.verify_ssl = False

client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=config,
    )
)

In [ ]:
core_api = k8s.CoreV1Api(k8s.ApiClient(config))


def get_container_startup_time(job_name):
    """Time between pod creation and training container start."""
    pods = core_api.list_namespaced_pod(
        namespace=parameters["namespace"],
        label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
    )
    for pod in pods.items:
        created = pod.metadata.creation_timestamp
        for status in pod.status.container_statuses or []:
            if status.name != "node":
                continue
            state = status.state.terminated or status.state.running
            if state and state.started_at:
                delta = (state.started_at - created).total_seconds()
                print(f"  {pod.metadata.name}:")
                print(f"    created:   {created.isoformat()}")
                print(f"    started:   {state.started_at.isoformat()}")
                print(f"    container startup: {delta:.1f}s")

## Job submission helpers

`TransformersTrainer` does not leave the training script's `SFTConfig` alone. It
patches `Trainer.__init__` and overwrites `output_dir`, `save_strategy`, and
`save_total_limit` with its own values. Two consequences shape the code below:

- **Checkpoint location comes from the `pvc://` URL, not from `SFTConfig`.** The
  PVC is mounted at `/mnt/kubeflow-checkpoints` and the URL path becomes a
  subdirectory of it, so `pvc://<pvc>/<run>/checkpoints` is what puts checkpoints
  where `wait_for_checkpoint` looks for them.
- **Save cadence comes from `PeriodicCheckpointConfig`.** Without it the SDK
  substitutes `save_strategy="epoch"` and discards `save_steps`, so the
  step checkpoint the notebook waits for is never written.

`enable_jit_checkpoint` is switched off because it is force-enabled by
`output_dir`. Left on, deleting the job triggers a checkpoint at the kill step,
and the restart would resume from there instead of from the step checkpoint.

In [ ]:
import os
import shutil
import time

from kubeflow.trainer.options import (
    ContainerOverride,
    Name,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubeflow_trainer_api.models import IoK8sApimachineryPkgApiResourceQuantity

PACKAGES = ["datasets", "peft", "trl", "nvidia-ml-py"]

WORLD_SIZE = parameters["num_nodes"] * parameters["gpus"]
RUN_NAME = f"phase3-restart-{WORLD_SIZE}gpu"
WORKBENCH_MOUNT = Path("/opt/app-root/src/elastic-scaling-shared")
RUN_DIR = WORKBENCH_MOUNT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"


def pod_overrides():
    return PodTemplateOverrides(
        PodTemplateOverride(
            target_jobs=["node"],
            spec=PodSpecOverride(
                node_selector={"kubernetes.io/hostname": parameters["target_node"]},
                volumes=[
                    {
                        "name": "dshm",
                        "emptyDir": {
                            "medium": "Memory",
                            "sizeLimit": IoK8sApimachineryPkgApiResourceQuantity("16Gi"),
                        },
                    },
                    {
                        "name": "hf-token",
                        "secret": {"secretName": parameters["hf_secret_name"]},
                    },
                ],
                containers=[
                    ContainerOverride(
                        name="node",
                        volume_mounts=[
                            {"name": "dshm", "mountPath": "/dev/shm"},
                            {"name": "hf-token", "mountPath": "/mnt/hf-token", "readOnly": True},
                        ],
                    ),
                ],
            ),
        )
    )


def submit_job(name, **overrides):
    trainer = TransformersTrainer(
        func=train_func,
        func_args={**parameters, **overrides},
        num_nodes=parameters["num_nodes"],
        resources_per_node={"nvidia.com/gpu": parameters["gpus"]},
        packages_to_install=PACKAGES,
        output_dir=f"pvc://{parameters['pvc_name']}/{RUN_NAME}/checkpoints",
        periodic_checkpoint_config=PeriodicCheckpointConfig(
            save_strategy="steps",
            save_steps=parameters["save_steps"],
            save_total_limit=3,
        ),
    )
    trainer.enable_jit_checkpoint = False
    runtime = client.backend.get_runtime("torch-distributed")
    job_name = client.train(
        trainer=trainer,
        runtime=runtime,
        options=[pod_overrides(), Name(name)],
    )
    print(f"Submitted {job_name} ({WORLD_SIZE} GPUs)")
    return job_name


def watch_job(name, poll_s=15, timeout_s=7200):
    seen, start = None, time.time()
    while time.time() - start < timeout_s:
        status = client.get_job(name).status
        if status != seen:
            print(f"  [{int(time.time() - start):5d}s] {status}")
            seen = status
        if status in ("Complete", "Failed"):
            return status
        time.sleep(poll_s)
    return "Timeout"


def wait_for_checkpoint(step, poll_s=1, timeout_s=3600):
    """Wait until a checkpoint directory for the given step exists on the PVC.

    Reads the directory with os.listdir rather than testing the child path with
    Path.exists: this volume has served stale attributes, and a real readdir
    defeats a cached negative lookup.
    """
    target = f"checkpoint-{step}"
    start = time.time()
    while time.time() - start < timeout_s:
        try:
            if target in os.listdir(CHECKPOINT_DIR):
                print(f"Checkpoint found: {CHECKPOINT_DIR / target}")
                return True
        except FileNotFoundError:
            pass
        time.sleep(poll_s)
    print(f"Timeout waiting for {CHECKPOINT_DIR / target}")
    return False


def wait_for_pods_gone(job_name, poll_s=5, timeout_s=300):
    """Wait until the job's pods are gone so its GPUs are free to reschedule."""
    start = time.time()
    while time.time() - start < timeout_s:
        pods = core_api.list_namespaced_pod(
            namespace=parameters["namespace"],
            label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
        )
        if not pods.items:
            print(f"Pods released after {int(time.time() - start)}s")
            return True
        time.sleep(poll_s)
    print(f"Timeout waiting for {job_name} pods to terminate")
    return False


def clean_checkpoints():
    """Remove checkpoints from the run directory and any stray at the PVC root."""
    for label, paths in [
        (str(CHECKPOINT_DIR), sorted(CHECKPOINT_DIR.glob("checkpoint-*"))),
        (f"{WORKBENCH_MOUNT} (shared PVC root)", sorted(WORKBENCH_MOUNT.glob("checkpoint-*"))),
    ]:
        if not paths:
            print(f"No checkpoints in {label}")
            continue
        print(f"Removing {len(paths)} checkpoint(s) from {label}:")
        for path in paths:
            shutil.rmtree(path)
            print(f"  {path.name}")

## Run the experiment

One block runs the whole measurement:

1. Clear stale checkpoints so the fresh run cannot auto-resume
2. Submit the fresh job and wait for `checkpoint-100`
3. Record container startup, then delete the job to simulate Kueue preemption
4. Wait for the GPUs to be released, submit the restart job, and record its
   startup timeline and logs

In [ ]:
clean_checkpoints()

fresh_job = submit_job("phase3-fresh", max_steps=1000)
if not wait_for_checkpoint(parameters["save_steps"]):
    raise RuntimeError("checkpoint never appeared, aborting")

print("\nFresh run container startup:")
get_container_startup_time(fresh_job)

client.delete_job(name=fresh_job)
print(f"\nDeleted {fresh_job} (simulated preemption)")
wait_for_pods_gone(fresh_job)

restart_job = submit_job("phase3-restart")
restart_status = watch_job(restart_job)
print(f"Restart result: {restart_status}")

print("\nRestart logs:")
for line in client.get_job_logs(restart_job, follow=False):
    if "[phase3]" in line:
        print(line, end="")

print("\nRestart container startup:")
get_container_startup_time(restart_job)

## Results

Read the restart startup timeline from the PVC.

In [ ]:
import json

import pandas as pd

timing_file = RUN_DIR / "startup_timing.json"
if timing_file.exists():
    timing = json.loads(timing_file.read_text())
    print(f"Run: {timing['run_name']}")
    print(f"Resumed from: {timing['resumed_from']}")
    print(f"Total training time: {timing['total_train_time_s']}s")
    print()
    print("Startup timeline:")
    df_timeline = pd.DataFrame(timing["timeline"])
    df_timeline
else:
    print(f"No timing file found at {timing_file}")

In [ ]:
# Check loss values after restart - does the loss look normal?
if timing_file.exists():
    losses = timing.get("losses", [])
    if losses:
        df_loss = pd.DataFrame(losses)[["step", "loss"]]
        print(f"Loss values after restart ({len(losses)} entries):")
        print(df_loss.to_string(index=False))
    else:
        print("No loss values recorded")

## Save & Cleanup

In [ ]:
out = Path("../results/phase3")
out.mkdir(parents=True, exist_ok=True)

if timing_file.exists():
    shutil.copy(timing_file, out / "startup_timing.json")
    print(f"Saved to {out.resolve()}")

In [ ]:
for job_name in ["phase3-fresh", "phase3-restart"]:
    try:
        client.delete_job(name=job_name)
        print(f"Deleted {job_name}")
    except Exception as error:
        print(f"Skip {job_name}: {error}")